# 6. Resuming and Persisting Runs

Every notebook so far trained and evaluated within one continuous Python
session. This notebook covers picking work back up in a *new* process —
after a kernel restart, on a different machine, or days later.


> **Prerequisites**
> - STP-Bench installed (`bash scripts/create_env.sh`, see the
>   [README](../README.md#installation)).
> - Benchmark data downloaded for the dataset(s) used below (see
>   [README — Benchmark Data](../README.md#benchmark-data)), or your own
>   dataset added following
>   [docs/guide.md — Adding a New Dataset](../docs/guide.md#adding-a-new-dataset).
> - Run this notebook from the repo root, or pass `repo_root=` explicitly to
>   `STPred(...)`.


## Resuming a known training run

`STPred.from_run(...)` reconstructs the checkpoint-resolution state that
`predict()`/`evaluate()` need, without re-running `train()` — as long as the
checkpoint files themselves are still on disk.


In [ ]:
from stpbench import STPred

stp = STPred.from_run(
    data="ncche/xenium",
    models=["StNet"],
    timestamp="2026-05-18-12-00-00",   # omit to auto-pick each model's latest run
)
stp.evaluate_external(data="hest/LUAD")


`timestamp` accepts a single string (used for every model), a
`{model: timestamp}` dict, or `None` (auto-picks the latest run directory
per model under `<log_path>/<data>/<model>/`).


## Persisting full workflow state

`save_state()` writes everything needed to reconstruct not just the
checkpoint, but the whole `STPred` instance's accumulated state: constructor
settings, `internal_data`/`external_data`, results so far, checkpoint
timestamps, `last_output_dir`, and more.


In [ ]:
stp.save_state("logs/my_stpred_state.yaml")


In [ ]:
stp2 = STPred(models=["StNet"])
stp2.load_state("logs/my_stpred_state.yaml")

stp2.predict(data="cptac/xenium")   # picks up exactly where stp left off


## Combining both in one call

`STPred.from_run(..., state_path=...)` combines construction and
`load_state()` in a single call — the shortest path from "cold process" to
"ready to call `evaluate_external()`/`predict()`/`downstream()`":


In [ ]:
stp3 = STPred.from_run(
    data="ncche/xenium",
    models=["StNet"],
    state_path="logs/my_stpred_state.yaml",
)


## Where this fits in a real workflow

A common pattern: train on one long-running job (notebook 2), then in a
separate, later session — possibly on a different machine with access to
the same `logs/`/data directories — resume via `from_run()` and run
`evaluate_external()` (notebook 2), `predict()` (notebook 3), or
`downstream()` (notebook 5) against new data, with no need to keep the
original training process alive or re-specify every setting by hand.


## Extending STP-Bench further

For extending the benchmark itself — adding a new dataset or a new model —
see
[docs/guide.md — Adding a New Dataset](../docs/guide.md#adding-a-new-dataset)
and
[docs/guide.md — Adding a New Model](../docs/guide.md#adding-a-new-model).
